# Tool-Based Agent Pattern: Function Calling
### When the process needs to *act*

A RAG agent produces answers. A tool-based agent produces **changes** — it looks up data, runs calculations, enforces rules, and writes updates. The agent decides which function to call, constructs the arguments from natural language, and reasons over the result.

Use function calling when a process step needs to:
- **Execute a calculation** — pricing, eligibility scoring, proration
- **Perform a lookup** — account status, plan details, inventory check
- **Enforce a rule** — discount tiers, approval thresholds, validation logic

The key signal: success is a **transaction**, not a response.

### Architecture

```
User Query → Agent → LLM decides which function to call
                         │
                ┌────────┼─────────────────────┐
                ▼        ▼                     ▼
         lookup_customer calculate_upgrade  apply_loyalty_discount
                │        │                     │
                └────────┼─────────────────────┘
                         ▼
                Agent reasons over results → Response
```

### Business Use Case: AnyComp Telecom Customer Account Management

A customer contacts AnyComp Telecom to upgrade their mobile plan. The agent needs to look up their account, check plan options, calculate the upgrade cost, and apply a loyalty discount based on tenure. Each step is a function call — deterministic, bounded, and already implemented in the codebase.


| Framework | Approach |
|---|---|
| **Strands Agents** | `@tool` decorated functions, agent auto-selects and calls |


## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip install -q strands-agents>=1.34.0 strands-agents-tools boto3 "typing_extensions>=4.12.0"

## Mock Data

Simple in-memory data representing AnyComp Telecom's customer and plan systems.

In [ ]:
# ── Mock data: customers and plans ─────────────────────────────────────────

CUSTOMERS = {
    "CUST-001": {"name": "Maria Chen", "plan": "Basic", "tenure_years": 5, "monthly_bill": 45.00},
    "CUST-002": {"name": "James Wilson", "plan": "Standard", "tenure_years": 2, "monthly_bill": 65.00},
    "CUST-003": {"name": "Priya Patel", "plan": "Premium", "tenure_years": 8, "monthly_bill": 95.00},
}

PLANS = {
    "Basic":    {"price": 45.00, "data_gb": 5,   "features": ["Calls", "SMS"]},
    "Standard": {"price": 65.00, "data_gb": 20,  "features": ["Calls", "SMS", "5G", "Hotspot"]},
    "Premium":  {"price": 95.00, "data_gb": 100, "features": ["Calls", "SMS", "5G", "Hotspot", "International", "Streaming"]},
    "Ultra":    {"price": 120.00, "data_gb": -1,  "features": ["Calls", "SMS", "5G", "Hotspot", "International", "Streaming", "Unlimited Data"]},
}

LOYALTY_TIERS = {
    (0, 2): 0.0,     # < 2 years: no discount
    (2, 5): 0.10,    # 2-5 years: 10%
    (5, 10): 0.15,   # 5-10 years: 15%
    (10, 99): 0.20,  # 10+ years: 20%
}

print("Mock data loaded: 3 customers, 4 plans, 4 loyalty tiers")

## Sample Request

We'll ask each framework to handle the same customer request:

In [ ]:
REQUEST = "Customer CUST-001 wants to upgrade to the Premium plan. Look up their account, show them the plan details, calculate the cost, and apply any loyalty discount they qualify for."

REGION = 'us-east-1'

## Implementation 1: Strands Agents

Strands uses `@tool` decorated functions. The agent inspects the function signatures and docstrings to decide which tool to call and how to construct the arguments.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def lookup_customer(customer_id: str) -> str:
    """Look up a customer account by ID.
    Args:
        customer_id: The customer identifier (e.g. CUST-001)
    """
    customer = CUSTOMERS.get(customer_id)
    if not customer:
        return f"Customer {customer_id} not found."
    return str(customer)

@tool
def get_plan_details(plan_name: str) -> str:
    """Get details for a mobile plan.
    Args:
        plan_name: Plan name (Basic, Standard, Premium, Ultra)
    """
    plan = PLANS.get(plan_name)
    if not plan:
        return f"Plan {plan_name} not found. Available: {list(PLANS.keys())}"
    return str(plan)

@tool
def calculate_upgrade_cost(current_plan: str, target_plan: str) -> str:
    """Calculate the monthly cost difference for a plan upgrade.
    Args:
        current_plan: Customer's current plan name
        target_plan: Plan they want to upgrade to
    """
    current = PLANS.get(current_plan, {})
    target = PLANS.get(target_plan, {})
    if not current or not target:
        return "Invalid plan name(s)."
    diff = target['price'] - current['price']
    return f"Upgrade from {current_plan} (${current['price']}/mo) to {target_plan} (${target['price']}/mo): +${diff:.2f}/mo"

@tool
def apply_loyalty_discount(customer_id: str, monthly_amount: float) -> str:
    """Apply a loyalty discount based on customer tenure.
    Args:
        customer_id: The customer identifier
        monthly_amount: The monthly amount before discount
    """
    customer = CUSTOMERS.get(customer_id)
    if not customer:
        return "Customer not found."
    tenure = customer['tenure_years']
    discount_pct = 0.0
    for (low, high), pct in LOYALTY_TIERS.items():
        if low <= tenure < high:
            discount_pct = pct
            break
    discount_amt = monthly_amount * discount_pct
    final = monthly_amount - discount_amt
    return f"Tenure: {tenure} years. Discount: {discount_pct*100:.0f}% (-${discount_amt:.2f}/mo). Final: ${final:.2f}/mo"


strands_model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    region_name=REGION,
    max_tokens=4096,
)

strands_agent = Agent(
    model=strands_model,
    system_prompt=(
        "You are a customer service agent for AnyComp Telecom. "
        "Use your tools to look up accounts, check plans, calculate costs, and apply discounts. "
        "Always show the customer the full breakdown before confirming any changes."
    ),
    tools=[lookup_customer, get_plan_details, calculate_upgrade_cost, apply_loyalty_discount],
)

print("=" * 60)
print("STRANDS AGENTS")
print("=" * 60)
print(strands_agent(REQUEST))